**Dataset:** city_temperature.csv

**Context:** Climate Analysis

In this notebook, I conducted a time-series analysis using SQL to identify **seasonality, trends, and cumulative variations** in global temperature data.
The objective was to produce clean and interpretable outputs suitable for integration into dashboards and strategic environmental reports.

**Skills demonstrated:** Time-series analysis, cumulative metrics, trend analysis, SQL aggregation
**Language:** SQL

In [1]:
df_1 = _dntk.execute_sql(
  '--Análise exploratória para verificar quantos dias iguais a "0" existem na coluna "Day" e alguns exemplos de cidades\n--onde estes valores apareceram.\n\nSELECT\n    Day, \n    COUNT(*) AS qtd_registros, \n    MIN(Year) AS ano_min,\n    MAX(Year) AS ano_max, \n    ARRAY_AGG(DISTINCT City)[:5] AS exemplos_cidades\nFROM read_csv_auto(\'city_temperature.csv\')\nWHERE Day < 1 OR Day > 31\nGROUP BY Day\nORDER BY Day;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_1

,Day,qtd_registros,ano_min,ano_max,exemplos_cidades
0,0,8,2008,2016,"[Conakry, Lilongwe, Kampala, Guadalajara, Bissau]"


In [2]:
Questao_1 = _dntk.execute_sql(
  '-- Aqui foi criada uma tabela temporária com três colunas extras: "data_evento", "categoria" e "valor". A coluna \n-- "data_evento" foi transformada para o tipo "datetime".\n\nCREATE OR REPLACE TABLE base_city_temperature AS\nSELECT\n    MAKE_DATE(Year, Month, Day) AS data_evento, \n    Year, \n    Month, \n    Day, \n    CASE\n        WHEN AvgTemperature = -99 THEN NULL\n        ELSE AvgTemperature\n    END AS valor, \n    City AS categoria, \n    Region, \n    Country, \n    State\nFROM read_csv_auto(\'city_temperature.csv\')\nWHERE Day >= 1;\n\nSELECT * FROM base_city_temperature;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
Questao_1

,data_evento,Year,Month,Day,valor,categoria,Region,Country,State
0,1995-01-01,1995,1,1,64.2,Algiers,Africa,Algeria,None
1,1995-01-02,1995,1,2,49.4,Algiers,Africa,Algeria,None
2,1995-01-03,1995,1,3,48.8,Algiers,Africa,Algeria,None
3,1995-01-04,1995,1,4,46.4,Algiers,Africa,Algeria,None
4,1995-01-05,1995,1,5,47.9,Algiers,Africa,Algeria,None
...,...,...,...,...,...,...,...,...,...
2906314,2013-07-27,2013,7,27,82.4,San Juan Puerto Rico,North America,US,Additional Territories
2906315,2013-07-28,2013,7,28,81.6,San Juan Puerto Rico,North America,US,Additional Territories
2906316,2013-07-29,2013,7,29,84.2,San Juan Puerto Rico,North America,US,Additional Territories
2906317,2013-07-30,2013,7,30,83.8,San Juan Puerto Rico,North America,US,Additional Territories


In [3]:
Questao_2 = _dntk.execute_sql(
  '-- Identificação da quantidade de dias como intervalo entre um evento e outro (média de temperatura). As categorias\n-- (cidades) foram agrupadas de forma a podermos visualizar mais claramente os intervalos internos.\n\nCREATE OR REPLACE TEMP TABLE city_event_intervals AS\nSELECT\n    categoria,                 \n    data_evento,                \n    LAG(data_evento) OVER (\n        PARTITION BY categoria\n        ORDER BY data_evento\n    ) AS data_anterior,         \n    DATE_DIFF(\'day\', \n        LAG(data_evento) OVER (PARTITION BY categoria ORDER BY data_evento),\n        data_evento\n    ) AS intervalo_dias         \nFROM base_city_temperature\nWHERE data_evento IS NOT NULL;\n\nSELECT * FROM city_event_intervals;\n\n\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
Questao_2

,categoria,data_evento,data_anterior,intervalo_dias
0,Chattanooga,1995-01-01,NaT,NaN
1,Chattanooga,1995-01-02,1995-01-01,1.0
2,Chattanooga,1995-01-03,1995-01-02,1.0
3,Chattanooga,1995-01-04,1995-01-03,1.0
4,Chattanooga,1995-01-05,1995-01-04,1.0
...,...,...,...,...
2906314,Pittsburgh,1995-07-11,1995-07-10,1.0
2906315,Pittsburgh,1995-07-12,1995-07-11,1.0
2906316,Pittsburgh,1995-07-13,1995-07-12,1.0
2906317,Pittsburgh,1995-07-14,1995-07-13,1.0


In [4]:
Questao_3 = _dntk.execute_sql(
  '-- No código abaixo foi criada uma tabela "temp_semanal" com uma coluna chamada "semana" cujos valores são \n-- representados pela primeira segunda feira de cada semana (através da função "DATE_TRUNC") e outra coluna com a \n-- média de temperatura semanal (através da função de agregação "AVG"). Essa mesma logística foi aplicada para as\n-- granularidades mensal e trimestral.\n\nCREATE OR REPLACE TABLE temp_semanal AS\nSELECT\n    categoria, \n    DATE_TRUNC(\'week\', data_evento) AS semana, \n    AVG(valor) AS temp_media_semanal\nFROM base_city_temperature\nGROUP BY categoria, DATE_TRUNC(\'week\', data_evento);\n\nCREATE OR REPLACE TABLE temp_mensal AS\nSELECT\n    categoria, \n    DATE_TRUNC(\'month\', data_evento) AS mes, \n    AVG(valor) AS temp_media_mensal\nFROM base_city_temperature\nGROUP BY categoria, DATE_TRUNC(\'month\', data_evento);\n\nCREATE OR REPLACE TABLE temp_trimestral AS\nSELECT\n    categoria, \n    DATE_TRUNC(\'quarter\', data_evento) AS trimestre, \n    AVG(valor) AS temp_media_trimestral\nFROM base_city_temperature\nGROUP BY categoria, DATE_TRUNC(\'quarter\', data_evento);\n\n-- No código abaixo foram utilizados "LEFT JOINs" nas três tabelas criadas acima onde a categoria ou seja, a cidade\n-- é Paris. As médias semanais, mensais e trimestrais podem ser visualizadas no dataframe abaixo. Além disso, para \n-- facilitar a leitura e implementação do código, foram criados alias para as tabelas. \n\nSELECT\n    s.semana, \n    s.temp_media_semanal, \n    m.temp_media_mensal, \n    t.temp_media_trimestral\nFROM temp_semanal s\nLEFT JOIN temp_mensal m\n    ON m.categoria = s.categoria AND DATE_TRUNC(\'month\', s.semana) = m.mes\nLEFT JOIN temp_trimestral t\n    ON t.categoria = s.categoria AND DATE_TRUNC(\'quarter\', s.semana) = t.trimestre\nWHERE s.categoria = \'Paris\'\nORDER BY s.semana;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
Questao_3

,semana,temp_media_semanal,temp_media_mensal,temp_media_trimestral
0,1994-12-26,37.400000,NaN,NaN
1,1995-01-02,34.128571,42.158065,45.327778
2,1995-01-09,42.385714,42.158065,45.327778
3,1995-01-16,44.928571,42.158065,45.327778
4,1995-01-23,47.942857,42.158065,45.327778
...,...,...,...,...
1320,2020-04-13,56.828571,57.176667,56.430233
1321,2020-04-20,60.528571,57.176667,56.430233
1322,2020-04-27,54.442857,57.176667,56.430233
1323,2020-05-04,58.200000,54.707692,56.430233


In [5]:
Questao_4 = _dntk.execute_sql(
  '-- No código abaixo utilizamos funções de janelamento para retornar as médias da temperatura (valor) dos últimos\n-- 7 dias (incluindo o dia atual) e dos últimos 30 dias (incluindo o atual). Para a querie final foi utilizada a  \n-- categoria (cidade) Paris. \n\nCREATE OR REPLACE TABLE tendencia_movel AS\nSELECT\n    categoria, \n    data_evento, \n    valor, \n\nAVG(valor) OVER(\n    PARTITION BY categoria\n    ORDER BY data_evento\n    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW\n) AS media_movel_7d, \n\nAVG(valor) OVER (\n    PARTITION BY categoria\n    ORDER BY data_evento\n    ROWS BETWEEN 29 PRECEDING AND CURRENT ROW\n) AS media_movel_30d\n\nFROM base_city_temperature\nWHERE valor IS NOT NULL\nORDER BY categoria, data_evento;\n\n\nSELECT\n    data_evento, \n    valor, \n    media_movel_7d, \n    media_movel_30d\nFROM tendencia_movel\nWHERE categoria = \'Paris\'\nORDER BY data_evento DESC\nLIMIT 50;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
Questao_4

,data_evento,valor,media_movel_7d,media_movel_30d
0,2020-05-13,43.6,54.785714,56.616667
1,2020-05-12,49.3,57.085714,56.976667
2,2020-05-11,50.0,56.957143,57.386667
3,2020-05-10,60.1,58.200000,57.856667
4,2020-05-09,58.9,57.600000,57.970000
5,2020-05-08,62.1,56.842857,58.096667
6,2020-05-07,59.5,55.314286,58.083333
7,2020-05-06,59.7,54.200000,58.060000
8,2020-05-05,48.4,53.014286,57.790000
9,2020-05-04,58.7,54.542857,58.056667


In [6]:
Questao_5 = _dntk.execute_sql(
  '-- No primeiro bloco de código foi criada uma tabela "evolucao_mensal" com a media mensal de cada categoria, ou seja, \n-- de cada cidade. A função DATE_TRUNC() foi utilizada para "congelar" os registros da coluna "data_evento" no primeiro\n-- dia do mês em questão e o AVG() foi utilizado no GROUP BY e retornou a média de cada mes de cada cidade. \n\n\nCREATE OR REPLACE TABLE evolucao_mensal AS\nSELECT \n    categoria, \n    DATE_TRUNC(\'month\', data_evento) AS mes, \n    AVG(valor) AS media_mensal\nFROM base_city_temperature\nWHERE valor IS NOT NULL\nGROUP BY categoria, DATE_TRUNC(\'month\', data_evento)\nORDER BY categoria, mes;\n\n-- No segundo bloco de código a função LAG() foi utilizada para trazer a média mensal do mês anterior e a partir disso\n-- uma operação matemática "(media_mensal - media_mes_anterior) / media_mes_anterior * 100" foi feita com o objetivo\n-- de trazer a média percentual de um mês para o outro.\n\nCREATE OR REPLACE TABLE variacao_mensal AS\nSELECT\n    categoria, \n    mes, \n    media_mensal, \n    LAG(media_mensal) OVER (\n        PARTITION BY categoria\n        ORDER BY mes\n    ) AS media_mes_anterior, \n    ROUND(\n        (media_mensal - LAG(media_mensal) OVER (PARTITION BY categoria ORDER BY mes))\n        / LAG(media_mensal) OVER (PARTITION BY categoria ORDER BY mes) * 100, \n        2\n    ) AS variacao_percentual\nFROM evolucao_mensal\nORDER BY categoria, mes;\n\n-- No terceiro bloco de código a cidade "Paris" foi escolhida para podermos observar a media mensal, a media do mês\n-- anterior e a variação percentual dos meses. Quando a variação é positiva, podemos observar que a temperatura do \n-- mês corrente é mais alta, ou seja, mais quente e quando é negativa, acontece o oposto.\n\nSELECT *\nFROM variacao_mensal\nWHERE categoria = \'Paris\'\nORDER BY mes\nLIMIT 20;\n\n\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
Questao_5

,categoria,mes,media_mensal,media_mes_anterior,variacao_percentual
0,Paris,1995-01-01,42.158065,NaN,NaN
1,Paris,1995-02-01,48.160714,42.158065,14.24
2,Paris,1995-03-01,45.938710,48.160714,-4.61
3,Paris,1995-04-01,51.226667,45.938710,11.51
4,Paris,1995-05-01,60.041935,51.226667,17.21
5,Paris,1995-06-01,63.740000,60.041935,6.16
6,Paris,1995-07-01,73.703226,63.740000,15.63
7,Paris,1995-08-01,72.654839,73.703226,-1.42
8,Paris,1995-09-01,60.520000,72.654839,-16.70
9,Paris,1995-10-01,60.220000,60.520000,-0.50


In [7]:
Questao_6 = _dntk.execute_sql(
  '-- Como primeira etapa do código abaixo, foi tirada a média da temperatura (valor) de cada mês de cada cidade \n-- (categoria) e depois foi utilizada a função de janelamento OVER(ROWS BETWEEN x PRECEDING ...) para retornar a \n-- média móvel de um período de 3 meses e outra de um período de 5 meses. \n\nCREATE OR REPLACE TABLE rolling_mensal AS\nSELECT\n    categoria, \n    DATE_TRUNC(\'month\', data_evento) AS mes, \n    AVG(valor) AS media_mensal, \n    AVG(AVG(valor)) OVER(\n        PARTITION BY categoria\n        ORDER BY DATE_TRUNC(\'month\', data_evento)\n        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW\n    ) AS media_3m, \n\n    AVG(AVG(valor)) OVER(\n        PARTITION BY categoria\n        ORDER BY DATE_TRUNC(\'month\', data_evento)\n        ROWS BETWEEN 4 PRECEDING AND CURRENT ROW\n    ) AS media_5m\nFROM base_city_temperature\nWHERE valor IS NOT NULL\nGROUP BY categoria, DATE_TRUNC(\'month\', data_evento)\nORDER BY categoria, mes;\n\nSELECT * FROM rolling_mensal;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
Questao_6

,categoria,mes,media_mensal,media_3m,media_5m
0,Abidjan,1995-01-01,79.916129,79.916129,79.916129
1,Abidjan,1995-02-01,82.614286,81.265207,81.265207
2,Abidjan,1995-03-01,82.545161,81.691859,81.691859
3,Abidjan,1995-04-01,83.320000,82.826482,82.098894
4,Abidjan,1995-05-01,82.403226,82.756129,82.159760
...,...,...,...,...,...
92908,Zurich,2020-01-01,36.683871,39.151864,45.913677
92909,Zurich,2020-02-01,43.165517,39.565925,42.727448
92910,Zurich,2020-03-01,42.645161,40.831516,40.653254
92911,Zurich,2020-04-01,55.076667,46.962448,43.283921


In [8]:
Questao_7 = _dntk.execute_sql(
  '-- No código abaixo foi utilizada a função SUM para somar todas as temperaturas que passaram dos 65F e que foram\n-- abaixo dos 65F. Aqui este valor foi considerado como o "confortável", por isso está sendo utilizado como referencia. 65F\n-- Alem disso, o EXTRACT(YEAR FROM...) foi utilizado para reiniciar a cada ano. \n\nCREATE OR REPLACE TEMP TABLE acumulado_degree_days AS\nSELECT\n    categoria,\n    data_evento,\n    valor AS temp_fahrenheit,\n\n\n    SUM(\n        CASE WHEN valor > 65 THEN valor - 65 ELSE 0 END\n    ) OVER (\n        PARTITION BY categoria, EXTRACT(YEAR FROM data_evento)\n        ORDER BY data_evento\n        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW\n    ) AS acima_65,\n\n   \n    SUM(\n        CASE WHEN valor < 65 THEN 65 - valor ELSE 0 END\n    ) OVER (\n        PARTITION BY categoria, EXTRACT(YEAR FROM data_evento)\n        ORDER BY data_evento\n        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW\n    ) AS abaixo_65\n\nFROM base_city_temperature\nWHERE data_evento IS NOT NULL\n  AND valor IS NOT NULL\nORDER BY categoria, data_evento;\n\n\nSELECT \n    data_evento, \n    temp_fahrenheit, \n    acima_65, \n    abaixo_65\nFROM acumulado_degree_days\nWHERE categoria = \'Paris\'\n  AND EXTRACT(YEAR FROM data_evento) = 2010\nORDER BY data_evento;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
Questao_7

,data_evento,temp_fahrenheit,acima_65,abaixo_65
0,2010-01-01,32.7,0.0,32.3
1,2010-01-02,31.7,0.0,65.6
2,2010-01-03,29.9,0.0,100.7
3,2010-01-04,23.6,0.0,142.1
4,2010-01-05,27.0,0.0,180.1
...,...,...,...,...
360,2010-12-27,30.5,351.7,5104.5
361,2010-12-28,33.8,351.7,5135.7
362,2010-12-29,37.4,351.7,5163.3
363,2010-12-30,35.2,351.7,5193.1


In [21]:
Questao_8 = _dntk.execute_sql(
  '-- Neste primeiro bloco de código foi calculada a média de cada mês de cada cidade (categoria) utilizando a função\n-- AVG() e o DATE_TRUNC para registrar o primeiro valor do mês. Uma tabela de base "media_mensal" foi criada.\n\nCREATE OR REPLACE TABLE media_mensal AS\nSELECT\n    categoria, \n    DATE_TRUNC(\'month\', data_evento) AS mes, \n    AVG(valor) AS media_temp_mensal\nFROM base_city_temperature\nWHERE valor IS NOT NULL\nGROUP BY categoria, DATE_TRUNC(\'month\', data_evento)\nORDER BY categoria, mes;\n\n-- Neste segundo bloco de código o LAG() foi utilizado para acessar tanto a média mensal do mês anterior quanto a \n-- média mensal do mês referente a 12 meses atrás e com isto foram criadas duas novas colunas chamadas media_ano_\n-- anterior e media_mes_anterior. \n\n\nCREATE OR REPLACE TABLE comparacoes AS\nSELECT\n    categoria, \n    mes, \n    media_temp_mensal, \n    LAG(media_temp_mensal, 12) OVER(\n        PARTITION BY categoria\n        ORDER BY mes\n    ) AS media_ano_anterior, \n    LAG(media_temp_mensal, 1) OVER (\n        PARTITION BY categoria\n        ORDER BY mes\n    ) AS media_mes_anterior, \n\n-- O próximo passo foi então utilizar o CASE WHEN para calcular a porcentagem referente à diferença de temperatura\n-- de um mês para o outro (variacao_mom) e de um mesmo mês de um ano para o outro (variacao_yoy).\n\n    CASE\n        WHEN LAG(media_temp_mensal, 12) OVER (PARTITION BY categoria ORDER BY mes) IS NOT NULL \n        THEN ROUND(\n            (media_temp_mensal - LAG(media_temp_mensal, 12) OVER (PARTITION BY categoria ORDER BY mes))\n            / LAG(media_temp_mensal, 12) OVER (PARTITION BY categoria ORDER BY mes) * 100, 2\n        )\n    END AS variacao_yoy,\n\n    CASE\n        WHEN LAG(media_temp_mensal, 1) OVER (PARTITION BY categoria ORDER BY mes) IS NOT NULL\n        THEN ROUND(\n            (media_temp_mensal - LAG(media_temp_mensal, 1) OVER (PARTITION BY categoria ORDER BY mes))\n            / LAG(media_temp_mensal, 1) OVER (PARTITION BY categoria ORDER BY mes) * 100, 2\n        )\n    END AS variacao_mom\n\nFROM media_mensal\nORDER BY categoria, mes;\n\n-- Bloco de código para visualizar a cidade \'Paris\' como referência.\n\nSELECT * FROM comparacoes\nWHERE categoria = \'Paris\'\nORDER BY mes\nLIMIT 24;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
Questao_8

,categoria,mes,media_temp_mensal,media_ano_anterior,media_mes_anterior,variacao_yoy,variacao_mom
0,Paris,1995-01-01,42.158065,NaN,NaN,NaN,NaN
1,Paris,1995-02-01,48.160714,NaN,42.158065,NaN,14.24
2,Paris,1995-03-01,45.938710,NaN,48.160714,NaN,-4.61
3,Paris,1995-04-01,51.226667,NaN,45.938710,NaN,11.51
4,Paris,1995-05-01,60.041935,NaN,51.226667,NaN,17.21
5,Paris,1995-06-01,63.740000,NaN,60.041935,NaN,6.16
6,Paris,1995-07-01,73.703226,NaN,63.740000,NaN,15.63
7,Paris,1995-08-01,72.654839,NaN,73.703226,NaN,-1.42
8,Paris,1995-09-01,60.520000,NaN,72.654839,NaN,-16.70
9,Paris,1995-10-01,60.220000,NaN,60.520000,NaN,-0.50


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b50e4344-4647-4a93-b699-42e32f41625a' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>